[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C43_Data_Engineering_Course/05_provenance_decontam/05_provenance_decontam.ipynb)

# 05 · 溯源与去污染（用 numpy/pandas/标准库从零实现 + 对拍 + 算账）

目标：把 **n-gram 去污染 → 数据 lineage DAG → 内容哈希快照 + 清单 → 审计报告** 从零实现，去污染检测**对拍已知污染真值**算召回，溯源/快照保证**可回溯、可复现**。

路线：n-gram 索引 + 去污染(对拍真值) → 子串包含检测 → lineage DAG(回溯/追踪) → 内容哈希 + 快照清单 → 审计报告 → ✏️ 练习 → 📖 答案 → 🧪 GPT-3/Dolma 胶囊。

> 心智模型：**去污染 = 把基准 n-gram 建哈希集合再扫训练集；lineage = 数据状态的 DAG；快照 = 内容哈希 + 不可变清单**。我们写算法*结构*与*正确性*，规模由账目推演。

> 这是本课**最后一个模块**——它把前四个模块（去重/加载/分词/过滤）做出来的数据变成**可信任**的数据。

## 1 · n-gram 抽取 + 基准索引（去污染的地基）

去污染的第一步：把所有**基准样本**的 n-gram 预建成一个**哈希集合**。
之后扫训练文档时，对每篇滑动取 n-gram、查这个集合即可——查集合是 O(1)，全程线性可流式。

In [ ]:
import numpy as np, hashlib, re, json
from collections import defaultdict
rng = np.random.default_rng(0)

def words(text):
    return re.findall(r'\w+', text.lower())

def ngrams(text, n=13):
    '''返回文本的所有连续 n-词-gram（字符串集合）。短于 n 则返回整串。'''
    w = words(text)
    if len(w) < n:
        return {' '.join(w)} if w else set()
    return {' '.join(w[i:i+n]) for i in range(len(w) - n + 1)}

def build_benchmark_index(benchmark_items, n=13):
    '''把所有基准样本的 n-gram 汇成一个集合（真实管线里用布隆过滤器省内存）。'''
    idx = set()
    for item in benchmark_items:
        idx |= ngrams(item, n)
    return idx

benchmark = [
    'what is the capital of france the answer is paris a well known european city',
    'solve for x if two x plus three equals eleven then x equals four exactly',
]
N = 8
bidx = build_benchmark_index(benchmark, n=N)
print(f'基准 {len(benchmark)} 题 -> {len(bidx)} 个 {N}-gram 进索引')
assert len(bidx) > 0
# 抽取正确性：基准题自己的 n-gram 必在索引里
assert ngrams(benchmark[0], N) <= bidx, '基准自身的 n-gram 应都在索引中'
print('✅ n-gram 抽取 + 基准索引正确：查询将是 O(1) 的集合命中')

## 2 · n-gram 去污染（对拍已知污染真值，算召回）

把去污染当成**严肃的检测问题**：造一批训练文档，其中一部分**已知**嵌入了基准题（污染真值），
另一部分是干净的。跑检测器，与真值对拍算**召回率**（漏没漏）和**精度**（误没误）。

In [ ]:
def is_contaminated(doc, benchmark_index, n=13):
    '''若文档任一 n-gram 命中基准索引，判为污染。返回 (是否污染, 命中的 n-gram 数)。'''
    hits = ngrams(doc, n) & benchmark_index
    return (len(hits) > 0), len(hits)

# 造训练集：清晰标注哪些被污染（真值）
clean_docs = [
    'machine learning systems process large amounts of data every single day now',
    'the weather forecast predicts heavy rain across the region this coming weekend',
    'distributed databases replicate data to survive hardware failures in production',
]
# 污染文档 = 一段普通文字里嵌入基准题（模拟论坛转载了考题）
contaminated_docs = [
    'here is a fun trivia question ' + benchmark[0] + ' hope you enjoyed it folks',
    'my math homework today was ' + benchmark[1] + ' and it took me a while',
]
train = clean_docs + contaminated_docs
truth_contaminated = set(range(len(clean_docs), len(train)))   # 真值：后两篇被污染

flagged = set()
for i, d in enumerate(train):
    bad, nhits = is_contaminated(d, bidx, n=N)
    if bad: flagged.add(i)
    print(f'doc {i}: {"污染" if bad else "干净"} (命中 {nhits} 个 {N}-gram)')

recall = len(flagged & truth_contaminated) / len(truth_contaminated)
precision = len(flagged & truth_contaminated) / max(len(flagged), 1)
print(f'\n召回={recall:.0%}  精度={precision:.0%}  (真值污染 {sorted(truth_contaminated)}, 检出 {sorted(flagged)})')
assert recall == 1.0, '嵌入的基准题应被全部检出（无漏报）'
assert precision == 1.0, '干净文档不应被误报'
print('✅ n-gram 去污染正确：召回 100%、精度 100% —— 把去污染当检测问题来量化')

## 3 · 子串包含检测 + n 的灵敏度权衡

n-gram 是局部检测；另一种是**整串包含**（基准题面是否作为子串出现在训练文档里）。
再做一个关键实验：**n 越小越易误报、越大越易漏报**——量化这个权衡，理解为何选 13-gram。

In [ ]:
def contains_check(doc, benchmark_items):
    '''归一化后判断任一基准题是否为文档的子串。'''
    nd = ' '.join(words(doc))
    for b in benchmark_items:
        if ' '.join(words(b)) in nd:
            return True
    return False

# 包含检测：嵌入了完整基准题的应判污染
assert contains_check(contaminated_docs[0], benchmark), '嵌入完整题面应被包含检测抓到'
assert not contains_check(clean_docs[0], benchmark), '干净文档不含基准题'
print('包含检测：精确但抓不住「只重叠一部分」的污染\n')

# n 的灵敏度：用一段含常见短语的干净文本，看不同 n 的误报
common = 'the result of the analysis shows that the model performs very well on the task'
# 把这段也塞进基准索引来源里，制造「常见短语」场景
bench_common = benchmark + ['the model performs very well on the task in most cases overall']
print(f"{'n':>4s} {'基准索引大小':>12s} {'common 文本是否误报':>18s}")
for n in [3, 6, 13]:
    idx_n = build_benchmark_index(bench_common, n=n)
    bad, _ = is_contaminated(common, idx_n, n=n)
    print(f'{n:>4d} {len(idx_n):>12d} {str(bad):>18s}')
# 小 n 易因常见短语误报；大 n 更稳健
bad3, _ = is_contaminated(common, build_benchmark_index(bench_common, 3), 3)
bad13, _ = is_contaminated(common, build_benchmark_index(bench_common, 13), 13)
assert bad3 and not bad13, 'n=3 应因常见短语误报，n=13 不应'
print('\n✅ n 越小越易误报（常见短语撞车），越大越易被改写绕过 —— 13-gram 是社区甜点')

## 4 · 数据 lineage：可回溯的 DAG

把每份数据记成一个节点：`(内容哈希, 步骤, 输入节点, 代码版本, 配置)`。连成 DAG，
就能从任一产物**向上回溯**到原始来源、或从某步 bug **向下追踪**所有受影响产物。

In [ ]:
class Lineage:
    def __init__(self):
        self.nodes = {}   # node_id -> dict(step, inputs, code_version, config, content_hash)
    def add(self, node_id, step, inputs, code_version, config, content_hash):
        self.nodes[node_id] = dict(step=step, inputs=list(inputs),
                                   code_version=code_version, config=config,
                                   content_hash=content_hash)
    def ancestors(self, node_id):
        '''向上回溯：该产物依赖的所有上游节点。'''
        seen, stack = set(), list(self.nodes[node_id]['inputs'])
        while stack:
            x = stack.pop()
            if x in seen: continue
            seen.add(x)
            stack += self.nodes.get(x, {}).get('inputs', [])
        return seen
    def descendants(self, node_id):
        '''向下追踪：依赖该节点的所有下游产物（某步 bug 的影响面）。'''
        out = set()
        for nid, nd in self.nodes.items():
            if node_id in self.ancestors(nid) or node_id in nd['inputs']:
                out.add(nid)
        return out

lin = Lineage()
lin.add('raw',      'crawl',         [],          'cc@2024.10', {}, 'a1b2')
lin.add('text',     'extract_text',  ['raw'],     'trafilatura@1.6', {'lang':'en'}, 'c3d4')
lin.add('filtered', 'quality_filter',['text'],    'pipeline@2.3', {'rules':'gopher'}, 'e5f6')
lin.add('deduped',  'dedup',         ['filtered'],'minhash@1.1', {'tau':0.8}, '7890')
lin.add('final',    'decontaminate', ['deduped'], 'ngram@1.0', {'n':13}, 'ffff')

anc = lin.ancestors('final')
print('final 的上游（回溯来源）:', sorted(anc))
assert anc == {'raw','text','filtered','deduped'}, 'final 应可回溯到原始 raw'
# 若 extract_text 这步有 bug，受影响的下游：
desc = lin.descendants('text')
print('text 步出 bug 时受影响的下游:', sorted(desc))
assert {'filtered','deduped','final'} <= desc, 'bug 影响面应含所有下游'
print('✅ lineage DAG 正确：可向上回溯来源、向下追踪 bug 影响面（增量重算的基础）')

## 5 · 内容哈希 + 可复现快照（清单 manifest）

给每个 shard 算 `SHA256`，把「shard → 哈希/大小」记进**清单**，再给清单算一个哈希作为**快照 ID**。
任何人拿到清单逐个校验，就能确认数据与你训练时**逐字节相同**。

In [ ]:
def content_hash(data_bytes):
    return hashlib.sha256(data_bytes).hexdigest()

def make_manifest(shards):
    '''shards: dict(shard_name -> bytes)。返回 (manifest, snapshot_id)。'''
    manifest = {}
    for name in sorted(shards):                       # 排序保证确定性
        b = shards[name]
        manifest[name] = {'sha256': content_hash(b), 'bytes': len(b)}
    # 快照 ID = 对整个清单（规范化 JSON）再哈希
    snapshot_id = content_hash(json.dumps(manifest, sort_keys=True).encode())
    return manifest, snapshot_id

def verify(shards, manifest):
    '''校验每个 shard 的哈希是否与清单一致。'''
    for name, b in shards.items():
        if content_hash(b) != manifest[name]['sha256']:
            return False
    return True

shards = {'shard_00': b'the quick brown fox', 'shard_01': b'jumps over lazy dog',
          'shard_02': b'machine learning data'}
manifest, snap_id = make_manifest(shards)
print('快照 ID:', snap_id[:16], '...')
for n, info in manifest.items():
    print(f'  {n}: sha256={info["sha256"][:12]}.. ({info["bytes"]} B)')

# 完整性：原样校验通过
assert verify(shards, manifest), '原始数据应校验通过'
# 不可变性：改一个字节 -> 校验失败、快照 ID 变
tampered = dict(shards); tampered['shard_00'] = b'the quick brown FOX'
assert not verify(tampered, manifest), '改一个字节应校验失败'
_, snap_id2 = make_manifest(tampered)
assert snap_id != snap_id2, '内容变了快照 ID 必须变（内容寻址）'
# 确定性：同样内容重算 -> 同样 ID
assert make_manifest(shards)[1] == snap_id, '同内容必得同快照 ID'
print('✅ 可复现快照正确：内容寻址 + 不可变 + 确定性 —— 可复现的物理基础')

## 6 · 数据审计报告：把可信变成可查的一页

把规模、来源构成、污染、PII 残留、可复现信息汇成一份**审计报告**（机器读 JSON + 人读摘要）。
这是「负责任地发布数据集」的交付物。

In [ ]:
import pandas as pd

def pii_residual_count(docs):
    '''抽样扫描残留 PII（邮箱为例，呼应模块 04）。'''
    email = re.compile(r'[\w.]+@[\w.]+')
    return sum(len(email.findall(d)) for d in docs)

def audit_report(train_docs, sources, benchmark_index, manifest, snapshot_id, n=8):
    n_contam = sum(is_contaminated(d, benchmark_index, n)[0] for d in train_docs)
    total_tokens = sum(len(words(d)) for d in train_docs)
    src_counts = pd.Series(sources).value_counts().to_dict()
    return {
        'n_documents': len(train_docs),
        'n_tokens': total_tokens,
        'source_mix': src_counts,
        'contamination_found': int(n_contam),
        'pii_residual': pii_residual_count(train_docs),
        'snapshot_id': snapshot_id[:16],
        'n_shards': len(manifest),
    }

audit_docs = train + ['contact me at john@example.com for details about the project']
sources = ['web','web','code','web','web','forum']
report = audit_report(audit_docs, sources, bidx, manifest, snap_id, n=N)
print('=== 数据审计报告 (机器读 JSON) ===')
print(json.dumps(report, ensure_ascii=False, indent=2))

assert report['n_documents'] == len(audit_docs)
assert report['contamination_found'] >= 2, '应报告出已知的污染文档'
assert report['pii_residual'] >= 1, '应报告出残留的邮箱 PII'
assert 'web' in report['source_mix']
print('\n✅ 审计报告正确：规模/来源/污染/PII/快照一页可查 —— 数据可信的证据')

---
## ✏️ 练习 1：带重叠比例阈值的去污染

硬阈值（命中 ≥1 个 n-gram 就判污染）对长文档太敏感。实现 `decontam_ratio(doc, bidx, n, min_ratio)`：
返回 `(是否污染, 重叠比例)`，其中**重叠比例 = 命中的 n-gram 数 / 文档总 n-gram 数**，
当比例 ≥ `min_ratio` 才判污染。空文档返回 `(False, 0.0)`。

In [ ]:
def decontam_ratio(doc, benchmark_index, n=8, min_ratio=0.1):
    # TODO: 取 doc 的 n-gram 集合 g；若为空返回 (False, 0.0)
    #       命中 = g & benchmark_index；ratio = len(命中)/len(g)
    #       返回 (ratio >= min_ratio, ratio)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 嵌入完整基准题的污染文档：重叠比例应较高
bad, ratio_bad = decontam_ratio(contaminated_docs[0], bidx, n=N, min_ratio=0.1)
clean, ratio_clean = decontam_ratio(clean_docs[0], bidx, n=N, min_ratio=0.1)
assert bad and ratio_bad > 0, '嵌入基准题的文档重叠比例应 >0 且判污染'
assert not clean and ratio_clean == 0.0, '干净文档重叠比例应为 0'
assert decontam_ratio('', bidx, n=N) == (False, 0.0), '空文档应返回 (False, 0.0)'
# 高阈值可放过「只重叠一点」的：把阈值设到 0.99，嵌在长文里的题可能被放过
_, r = decontam_ratio(contaminated_docs[0], bidx, n=N)
assert 0.0 < r <= 1.0
print(f'污染文档重叠比例={ratio_bad:.2f}, 干净={ratio_clean:.2f}')
print('✅ 练习 1 通过：带比例阈值的去污染（比硬阈值更鲁棒）')

## ✏️ 练习 2：lineage 回溯查询

实现 `root_sources(lineage, node_id)`：返回某产物的所有**根来源**（没有任何输入的上游节点，
即原始数据）。复用第 4 节的 `Lineage`。提示：在 `ancestors` 结果里筛出 `inputs == []` 的节点。

In [ ]:
def root_sources(lineage, node_id):
    # TODO: 取 lineage.ancestors(node_id)，筛出其中 inputs 为空的节点，返回集合
    #       （若 node_id 自己就是根，也要考虑——但本题只需返回上游里的根）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
roots = root_sources(lin, 'final')
assert roots == {'raw'}, 'final 的唯一根来源应是 raw（原始快照）'
# 给一个多源的例子
lin2 = Lineage()
lin2.add('web_raw', 'crawl', [], 'cc@1', {}, 'h1')
lin2.add('code_raw','clone', [], 'gh@1', {}, 'h2')
lin2.add('mixed',   'mix', ['web_raw','code_raw'], 'mix@1', {}, 'h3')
assert root_sources(lin2, 'mixed') == {'web_raw','code_raw'}, '应找出两个根来源'
print('✅ 练习 2 通过：能回溯任一产物的原始根来源')

## ✏️ 练习 3：快照差异（增量更新）

语料每月更新。实现 `snapshot_diff(manifest_old, manifest_new)`：对比两份清单，返回 dict
`{'added':[...], 'removed':[...], 'changed':[...]}`——新增的 shard、删除的 shard、
以及**名字相同但哈希不同**（内容变了）的 shard。各列表按名字排序。

In [ ]:
def snapshot_diff(manifest_old, manifest_new):
    # TODO: added = 在 new 不在 old 的名字；removed = 在 old 不在 new 的；
    #       changed = 两边都有但 sha256 不同的；都按 sorted 返回
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
m_old, _ = make_manifest({'a': b'1', 'b': b'2', 'c': b'3'})
m_new, _ = make_manifest({'b': b'2', 'c': b'CHANGED', 'd': b'4'})
diff = snapshot_diff(m_old, m_new)
assert diff['added'] == ['d'], 'd 是新增'
assert diff['removed'] == ['a'], 'a 被删除'
assert diff['changed'] == ['c'], 'c 内容变了（哈希不同）'
print('快照差异:', diff)
print('✅ 练习 3 通过：能算两快照的增量差异（只重算变化部分）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def decontam_ratio(doc, benchmark_index, n=8, min_ratio=0.1):
    g = ngrams(doc, n)
    if not g:
        return (False, 0.0)
    hits = g & benchmark_index
    ratio = len(hits) / len(g)
    return (ratio >= min_ratio, ratio)

In [ ]:
# 练习 2 参考答案
def root_sources(lineage, node_id):
    anc = lineage.ancestors(node_id)
    return {a for a in anc if not lineage.nodes.get(a, {}).get('inputs')}

In [ ]:
# 练习 3 参考答案
def snapshot_diff(manifest_old, manifest_new):
    old, new = set(manifest_old), set(manifest_new)
    added = sorted(new - old)
    removed = sorted(old - new)
    changed = sorted(k for k in (old & new)
                     if manifest_old[k]['sha256'] != manifest_new[k]['sha256'])
    return {'added': added, 'removed': removed, 'changed': changed}

---
## 🧪 真实数据胶囊：GPT-3 / Dolma 的去污染与审计算一笔账

用真实工程的公开做法与量级，算去污染的复杂度账与可信账：
① GPT-3 的 13-gram 去污染在万亿 token 上的扫描开销（线性，可行）；② 基准索引内存；③ 内容哈希快照的开销。

（带 try/except：本环境不联网，直接用内置的真实量级数字——GPT-3 论文 §污染分析、Dolma 工具包。）

In [ ]:
# 公开量级（约数）：GPT-3 13-gram 去污染 / Dolma 3T 语料
TRAIN_TOKENS = 1.5e13     # 训练集总 token（量级）
N_GRAM = 13               # GPT-3 用的 13-gram
N_BENCHMARKS = 30         # 评测基准数量（量级）
TOKENS_PER_BENCH = 1e5    # 每基准的 token 量级
SCAN_RATE = 1e7           # 单核每秒扫 ~1e7 token 取 n-gram + 查集合

# ① 扫描训练集：O(N)，线性一遍
scan_core_secs = TRAIN_TOKENS / SCAN_RATE
scan_days_1core = scan_core_secs / 86400
scan_hours_1024 = scan_core_secs / 1024 / 3600
print(f'① 13-gram 去污染扫描: 单核 {scan_days_1core:.1f} 天 / 1024 核 {scan_hours_1024:.1f} 小时 (O(N) 可流式)')

# ② 基准索引内存（所有基准的 n-gram 哈希集合）
bench_ngrams = N_BENCHMARKS * TOKENS_PER_BENCH      # ~n-gram 数量级
idx_bytes = bench_ngrams * 8                         # 每个 n-gram 哈希 ~8 字节
print(f'② 基准索引: {bench_ngrams:.1e} 个 n-gram, ~{idx_bytes/1e6:.0f} MB (布隆过滤器可再压)')

# ③ 内容哈希快照: SHA256 ~1 GB/s，扫一遍数据
DATA_TB = 50
hash_hours = DATA_TB * 1e12 / 1e9 / 3600
print(f'③ 50 TB 快照哈希: ~{hash_hours:.1f} 小时 (一遍 SHA256, 可并行)')

# 对比：朴素「每篇训练文档 vs 每个基准样本」两两比对
naive_ops = TRAIN_TOKENS * (N_BENCHMARKS * TOKENS_PER_BENCH)
print(f'\n朴素两两比对: {naive_ops:.1e} 次 -> 哈希集合把基准侧压成 O(1) 查询')
assert scan_core_secs < naive_ops / SCAN_RATE, '哈希索引应远快于两两比对'
assert idx_bytes < 1e9, '基准索引应能放进单机内存（MB 级）'
print('\n账目结论：去污染/快照都是 O(N) 线性、相对训练几乎免费 —— 没有不做的理由。')

**🧪 胶囊练习**：实现 `decontam_cost(train_tokens, scan_rate, cores, eff=0.8)`：估算去污染扫描的耗时（秒）。有效吞吐 = `scan_rate * cores * eff`，返回 `train_tokens / 有效吞吐`。

In [ ]:
def decontam_cost(train_tokens, scan_rate, cores, eff=0.8):
    # TODO: 返回 train_tokens / (scan_rate * cores * eff)
    raise NotImplementedError

In [ ]:
# 自测
secs = decontam_cost(1.5e13, 1e7, 1024)
assert abs(secs - 1.5e13 / (1e7 * 1024 * 0.8)) < 1e-6
# 1024 核应把单核的几天压到小时级
secs_1 = decontam_cost(1.5e13, 1e7, 1)
assert secs_1 / secs > 800, '1024 核应带来近 1024x（打折后）加速'
print(f'1.5e13 token 去污染: 1024 核 {secs/3600:.1f} 小时')
print('✅ 胶囊练习通过：去污染是 O(N) 线性、可并行')

In [ ]:
# 📖 胶囊参考答案
def decontam_cost(train_tokens, scan_rate, cores, eff=0.8):
    return train_tokens / (scan_rate * cores * eff)

---
## 🔧 旁注：真实管线里的溯源去污染长什么样

本课的小模拟，在 GPT-3 / Dolma / FineWeb 等真实工程里对应：

- **n-gram 去污染**：GPT-3 用 13-gram 重叠从训练集找出与各基准重叠的样本并分析影响；Dolma 工具包内置去污染步骤，跑在 Spark/Ray 上、基准 n-gram 用布隆过滤器省内存。
- **lineage**：生产管线用 DAG 编排工具（Airflow/Dagster/自研）记录每步的代码版本、配置、输入输出哈希；出问题时顺图回溯/追踪，bug 只重算受影响下游。
- **快照 + 清单**：Dolma/HF Hub 用内容哈希给数据集打不可变版本，清单（manifest）记录全部 shard 哈希；类比 Git commit / Docker digest 的内容寻址。
- **审计 + datasheet**：Dodge《Documenting C4》、Gebru《Datasheets》是范本——结构化报告进监控、自然语言 datasheet 给使用者知情。

你在 numpy/pandas 里验证过的检测、回溯、哈希、报告逻辑，可几乎一对一搬到真实工具链上。

### 小结 + 全课收官
**本模块**：
- 去污染 = 把基准 n-gram 建哈希集合再扫训练集（O(N) 线性）；当检测问题量化召回/精度，13-gram 是甜点。
- **lineage** = 数据状态的 DAG，可回溯来源、追踪 bug 影响面（增量重算的基础）。
- **内容哈希快照** = 内容寻址 + 不可变 + 确定性，是可复现的物理基础。
- **审计报告** = 把规模/来源/污染/PII/快照汇成可查证据；机器读 JSON + 人读 datasheet。

**🎓 全课收官**：六个模块讲的是同一件事——**规模如何改变正确的做法**。
- 01 去重：O(n²)→近 O(n)（MinHash+LSH）  
- 02 流式加载：全量打乱→shuffle 缓冲近似  
- 03 分词吞吐：串行→并行 + packing 消浪费  
- 04 质量过滤：贵的在前→便宜的在前的多阶段  
- 05 溯源去污染：人工核查→自动化检测 + 哈希快照  

每一条，答案都是用**线性/近线性、可流式、可并行、可验证**的工程，替换掉朴素但不可扩展的做法。这就是大规模数据工程的全部手艺。

回到 **[课程主页](../index.html)**，或与 C21（数据科学）、C08（训练系统）、C37（MLOps）配合，把数据→训练→运维的全链路打通。